<a href="https://colab.research.google.com/github/CMDPARZIVAL/ARP-Poisoning-Detector/blob/main/ARP_Poisoning_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scapy
!sudo apt-get update
!sudo apt-get install libpcap-dev

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libpcap-dev is already the newest version (1.10.1-4ubuntu1.22.04.1).
0 upgraded, 0 newly insta

Saving the ARP Detector in detector.py file


In [ ]:
%%writefile detector.py
# ... [the rest of your code] ...
import scapy.all as scapy
import argparse
import logging
import time

# ==========================================
# 1. Configuration & Setup
# ==========================================

# Set up logging to save alerts to a file
logging.basicConfig(
    filename="arp_alerts.log",
    level=logging.WARNING,
    format="%(asctime)s - %(message)s"
)

# The "Memory" Dictionary: Maps IP addresses to MAC addresses
ip_mac_map = {}

# ==========================================
# 2. Core Logic
# ==========================================

def process_packet(packet):
    """
    Analyzes captured packets to detect MAC conflicts.
    """
    # Verify the packet is an ARP Reply (op=2)
    if packet.haslayer(scapy.ARP) and packet[scapy.ARP].op == 2:

        src_ip = packet[scapy.ARP].psrc
        src_mac = packet[scapy.ARP].hwsrc

        # Scenario A: We have never seen this IP before
        if src_ip not in ip_mac_map:
            ip_mac_map[src_ip] = src_mac
            # Optional: Uncomment the line below to see normal network mapping in real-time
            # print(f"[*] Learned new host: {src_ip} -> {src_mac}")

        # Scenario B: We HAVE seen this IP, check for a MAC conflict
        elif ip_mac_map[src_ip] != src_mac:
            old_mac = ip_mac_map[src_ip]

            # Construct the alert message
            alert_msg = f"ARP POISONING DETECTED! IP {src_ip} changed from {old_mac} to {src_mac}"

            # Print a highly visible warning to the console
            print("\n" + "!" * 70)
            print(f"[ALERT] {alert_msg}")
            print("!" * 70 + "\n")

            # Save the alert to the log file
            logging.warning(alert_msg)

# ==========================================
# 3. User Interface & Execution
# ==========================================

def get_arguments():
    parser = argparse.ArgumentParser(description="Passive ARP Poisoning Detector")
    parser.add_argument("-i", "--interface", help="Specify the network interface (e.g., eth0, wlan0, Wi-Fi)")

    # parse_known_args() returns a tuple.
    # 'args' gets the -i flag if it exists, 'unknown' catches Colab's hidden -f flag safely.
    args, unknown = parser.parse_known_args()
    return args

def main():
    args = get_arguments()
    iface = args.interface

    print(r"""
       ___  ___  ___   ____       __          __
      / _ \/ _ \/ _ \ / __ \___  / /____ ____/ /____  ____
     / , _/ , _/ ___// /_/ / -_)/ __/ -_) __/ __/ _ \/ __/
    /_/|_/_/|_/_/   /_____/\__/ \__/\__/\__/\__/\___/_/

    """)
    print("[*] Initializing ARP Detector...")

    if iface:
        print(f"[*] Bound to interface: {iface}")
    else:
        print("[*] Bound to default system interface.")

    print("[*] Actively monitoring for MAC conflicts...")
    print("[*] Any detected attacks will be saved to 'arp_alerts.log'.")
    print("[*] Press Ctrl+C to exit.\n")

    try:
        # Start sniffing. We use store=False to prevent memory leaks over time.
        if iface:
            scapy.sniff(iface=iface, store=False, prn=process_packet)
        else:
            scapy.sniff(store=False, prn=process_packet)

    except KeyboardInterrupt:
        print("\n[*] User interrupted. Shutting down detector. Goodbye!")
    except PermissionError:
        print("\n[!] ERROR: You need root/Administrator privileges to run this script.")
    except Exception as e:
        print(f"\n[!] A fatal error occurred: {e}")

if __name__ == "__main__":
    main()

Overwriting detector.py


Running detector.py in the background

In [ ]:
!nohup python3 detector.py -i eth0 > /dev/null 2>&1 &

Sending fake Packets

In [ ]:
from scapy.all import sendp, Ether, ARP
import time

# We use a dummy IP so we don't break Colab's actual internet connection
target_ip = "10.99.99.99"
normal_mac = "11:22:33:44:55:66"
attacker_mac = "AA:BB:CC:DD:EE:FF"

print("[*] Sending 'Normal' ARP Reply (Building baseline)...")
# Ether() wraps it in a standard network frame, ARP() builds the payload
normal_pkt = Ether(dst="ff:ff:ff:ff:ff:ff") / ARP(op=2, psrc=target_ip, hwsrc=normal_mac)
sendp(normal_pkt, iface="eth0", verbose=False)

time.sleep(2) # Give the detector a moment to process

print("[*] Sending Malicious ARP Reply (POISONING)...")
attack_pkt = Ether(dst="ff:ff:ff:ff:ff:ff") / ARP(op=2, psrc=target_ip, hwsrc=attacker_mac)
sendp(attack_pkt, iface="eth0", verbose=False)

print("[*] Attack simulation complete!")

[*] Sending 'Normal' ARP Reply (Building baseline)...
[*] Sending Malicious ARP Reply (POISONING)...
[*] Attack simulation complete!


checking the logs

In [ ]:
!cat arp_alerts.log

[ALERT] ARP POISONING DETECTED! IP 10.99.99.99 changed from 11:22:33:44:55:66 to aa:bb:cc:dd:ee:ff
[ALERT] ARP POISONING DETECTED! IP 10.99.99.99 changed from 11:22:33:44:55:66 to aa:bb:cc:dd:ee:ff
[ALERT] ARP POISONING DETECTED! IP 10.99.99.99 changed from 11:22:33:44:55:66 to aa:bb:cc:dd:ee:ff
[ALERT] ARP POISONING DETECTED! IP 10.99.99.99 changed from 11:22:33:44:55:66 to aa:bb:cc:dd:ee:ff
[ALERT] ARP POISONING DETECTED! IP 10.99.99.99 changed from 11:22:33:44:55:66 to aa:bb:cc:dd:ee:ff
[ALERT] ARP POISONING DETECTED! IP 10.99.99.99 changed from 11:22:33:44:55:66 to aa:bb:cc:dd:ee:ff
[ALERT] ARP POISONING DETECTED! IP 10.99.99.99 changed from aa:bb:cc:dd:ee:ff to 11:22:33:44:55:66
2026-04-02 10:18:05,003 - ARP POISONING DETECTED! IP 10.99.99.99 changed from 11:22:33:44:55:66 to aa:bb:cc:dd:ee:ff
[ALERT] ARP POISONING DETECTED! IP 10.99.99.99 changed from 11:22:33:44:55:66 to aa:bb:cc:dd:ee:ff
[ALERT] ARP POISONING DETECTED! IP 10.99.99.99 changed from 11:22:33:44:55:66 to aa:bb:cc:d